In [1]:
import jax
import jax.numpy as jnp
from typing import Callable, Sequence, Any
from functools import partial

jax.config.update('jax_enable_x64', True)

In [2]:
def _get_zero_placeholder(func: Callable[..., jnp.ndarray]) -> Callable[..., jnp.ndarray]:
    """Returns a new function that returns zeros of the same shape as func."""
    def zero_placeholder_func(*args, **kwargs):
        return func(*args, **kwargs) * 0.0

    return zero_placeholder_func

def fill_intensity_matrix(
        intensity_matrix: Sequence[Sequence[Callable[..., jnp.ndarray]]],
        fill_func = Callable[..., jnp.ndarray]
) -> Sequence[Sequence[Callable[..., jnp.ndarray]]]:
    filled_matrix = [
        [
            (func if func is not None else fill_func)
            for func in row 
        ]
        for row in intensity_matrix
    ]
    
    return filled_matrix

def construct_func_from_intensity_matrix(
        intensity_matrix: Sequence[Sequence[Callable[..., jnp.ndarray]]]
) -> Callable[..., tuple[jnp.ndarray, jnp.ndarray]]:
    """
    Returns a new function that computes the outflow and inflow according to an
    n x n intensity matrix.
    """
    n = len(intensity_matrix)
    if not all(len(row) == n for row in intensity_matrix):
        raise ValueError("Intensity matrix must be square (n x n).")
    
    all_functions = (f for row in intensity_matrix for f in row)
    global_reference_func = next((f for f in all_functions if f is not None), None)
    
    if global_reference_func is None:
        raise ValueError("The intensity matrix contains only None entries. Cannot construct any function or determine output shape.")
    
    zero_func = _get_zero_placeholder(global_reference_func)
    intensity_matrix = fill_intensity_matrix(intensity_matrix, zero_func)
    
    @jax.jit
    def outflow_inflow(*args: Any, **kwargs: Any) -> tuple[jnp.ndarray, jnp.ndarray]:
        evaluated_matrix = [
            [func(*args, **kwargs)for func in row]
            for row in intensity_matrix
        ]

        outflow = [sum(row) for row in evaluated_matrix]
        outflow = jnp.stack(outflow, axis=-2)

        tranposed_matrix = list(zip(*evaluated_matrix))
        inflow = [jnp.stack(col, axis=-2) for col in tranposed_matrix]
        inflow = jnp.stack(inflow, axis=-3)       

        return outflow, inflow
    
    return outflow_inflow

@jax.jit
def next_p_j(p_j, outflow, inflow, step_size):
    dp_j = jnp.diff(p_j, axis=-1, prepend=0)
    outflow_integral = jnp.cumsum(outflow * dp_j, axis=-1)
    inflow_integral = jnp.sum(inflow * jnp.expand_dims(dp_j, axis=-3), axis=(-2,-1))
    inflow_integral = jnp.expand_dims(inflow_integral, axis=-1)
    
    p_j = p_j + jnp.expand_dims(step_size, axis=-1) * (inflow_integral - outflow_integral)
    p_j = jnp.roll(p_j, shift=1, axis=-1)
    p_j = p_j.at[..., 0].set(0)
    
    return p_j

@partial(jax.jit, static_argnames=['flow'])
def solve_p_j(p_j_0: jnp.ndarray, 
              step_sizes: jnp.ndarray, 
              flow: Callable[..., tuple[jnp.ndarray, jnp.ndarray]], 
              *args: jnp.ndarray,
              **kwargs: jnp.ndarray):

    def scan_p_j(carry, step_size):
        p_j, t = carry
        outflow, inflow = flow(t, *args, **kwargs)
        p_j = next_p_j(p_j, outflow, inflow, step_size)
        t += step_size

        return (p_j, t), p_j
    
    t0 = jnp.zeros_like(step_sizes[0])
    _, p_j = jax.lax.scan(scan_p_j, (p_j_0, t0), step_sizes)
    
    p_j = jnp.concatenate([jnp.expand_dims(p_j_0, axis=0), p_j])
    p_j = jnp.swapaxes(p_j, 0, 1)
    p_j = jnp.swapaxes(p_j, 1, 2)

    return p_j

@partial(jax.jit, static_argnames=['flow'])
def solve(step_sizes: jnp.ndarray, 
          flow: Callable[..., tuple[jnp.ndarray, jnp.ndarray]], 
          *args: jnp.ndarray, 
          **kwargs: jnp.ndarray):

    outflow, inflow = flow(0, *args, **kwargs)
    step_sizes = jnp.expand_dims(step_sizes, axis=-1)
    
    p_j_0 = jnp.zeros_like(outflow)
    p_j_0 = p_j_0.at[(..., 0, slice(None))].set(1.0)

    p_j = solve_p_j(p_j_0, step_sizes, flow, *args, **kwargs)
    
    return p_j


In [3]:
@jax.jit
def mu_1(t, u):
    return 0 * u + 0.15

@jax.jit
def mu_2(t, u):
    return 1.2 * jnp.exp(-0.05 * t + u * 0) * 0.25

In [ ]:
u = jnp.concatenate([
    jnp.arange(0, 1, 1 / 48),
    jnp.arange(1, 2, 1 / 24),
    jnp.arange(2, 5, 1 / 12),
    jnp.arange(5, 30 + 1, 1)
])

#u = jax.device_put(u, device=jax.devices()[0])

u2 = jnp.expand_dims(u, axis=0) + jnp.expand_dims(jnp.arange(0, 1, 0.0002), axis=1)
step_sizes = jnp.swapaxes(jnp.diff(u2), 1, 0)

intensity_matrix = [
    [None, mu_1, mu_1, mu_1, None],
    [None, None, mu_1, mu_1, mu_1],
    [None, None, None, None, None],
    [None, None, None, None, None],
    [None, None, None, None, None]
]

flow = construct_func_from_intensity_matrix(intensity_matrix)

In [8]:
%timeit -r 10 -n 10 solve(step_sizes, flow, u2).block_until_ready()

310 ms ± 11.5 ms per loop (mean ± std. dev. of 10 runs, 10 loops each)


In [7]:
%timeit -r 1 -n 10 solve(step_sizes, flow, u2).block_until_ready()

344 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 10 loops each)


In [9]:
%timeit -r 2 -n 100 solve(step_sizes, flow, u2).block_until_ready()

IndexError: index is out of bounds for axis 0 with size 0

In [ ]:
next_p_j(p_j, oo, ii, uu).shape

In [ ]:
jnp.cumsum(oo * dp_j, axis=-1)

In [ ]:
inflow_integral = jnp.sum(ii * jnp.expand_dims(dp_j, axis=-3), axis=(-2,-1))
inflow_integral = jnp.expand_dims(inflow_integral, axis=-1)

In [ ]:
inflow_integral

In [ ]:
dp_j = jnp.diff(p_j, axis=-1, prepend=0)
outflow_integral = jnp.cumsum(oo * dp_j, axis=-1)
inflow_integral = jnp.sum(ii * jnp.expand_dims(dp_j, axis=-3), axis=(-2,-1))
inflow_integral = jnp.expand_dims(inflow_integral, axis=-1)

(p_j + uu * (inflow_integral - outflow_integral))[0]

In [ ]:
def test_func(carry, step_size):
    print(step_size.shape)
    return carry, step_size

In [ ]:
final, hist = jax.lax.scan(test_func, 0, jnp.expand_dims(u3, axis=-1))